# LoRA で LLM を少ないメモリでファインチューニングする

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm/lora_finetune.ipynb)

NLP クックブック 13-14 ／ 大規模言語モデル (LLM)

> GPU ランタイム推奨（ランタイム → ランタイムのタイプを変更 → T4 GPU）

LLM を自分のタスク（ここでは文法誤り訂正）に適応させたい。全パラメータを更新するのは重いので、LoRA で一部だけ学習する。

In [ ]:
!pip install -q peft datasets accelerate "torchao>=0.16"   # Colab に入っている古い torchao (0.10) のままだと peft の import が失敗する

In [ ]:
import sentencepiece as spm
import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from peft import LoraConfig, get_peft_model
from tokenizers import models
from transformers import AutoModelForCausalLM, AutoTokenizer

name = "sbintuitions/sarashina2.2-0.5b-instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(name)
# transformers 5 系は、このモデルの SentencePiece (Unigram) を BPE に変換してしまい、学習時と違う分割になる。
# 元の tokenizer.model から語彙とスコアを読み、分割の方式だけを Unigram に戻す (特殊トークンやチャットテンプレートはそのまま)
sp = spm.SentencePieceProcessor(model_file=hf_hub_download(name, "tokenizer.model"))
tokenizer.backend_tokenizer.model = models.Unigram(
    [(sp.id_to_piece(i), sp.get_score(i)) for i in range(sp.get_piece_size())], unk_id=sp.unk_id(), byte_fallback=True)
BF16 = torch.cuda.get_device_capability()[0] >= 8      # bfloat16 は Ampere (A100, L4) 以降。T4 では float16 を使う
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto")

lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj"])   # アテンションの射影行列に低ランク行列を追加
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# 学習データ: 誤りを含む文 → 訂正文 (実際には数千〜数万ペアを用意する)
pairs = [("私は学校を行きます。", "私は学校に行きます。"), ("昨日は雨が降ります。", "昨日は雨が降りました。"),
         ("この本は面白いでした。", "この本は面白かったです。"), ("静かの部屋で勉強します。", "静かな部屋で勉強します。"),
         ("日本語を話すことが上手になりたい。", "日本語を話すのが上手になりたい。"), ("友達を会いました。", "友達に会いました。"),
         ("彼は背が高いだ。", "彼は背が高い。"), ("毎日、日本語が勉強します。", "毎日、日本語を勉強します。")] * 25
INSTRUCTION = "次の文の文法誤りを訂正してください。"

def encode(example):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n{example['src']}"}]
    prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    answer_ids = tokenizer(example["tgt"] + tokenizer.eos_token, add_special_tokens=False)["input_ids"]
    return {"input_ids": prompt_ids + answer_ids,
            "labels": [-100] * len(prompt_ids) + answer_ids}    # プロンプト部分は損失を計算しない

ds = Dataset.from_list([{"src": s, "tgt": t} for s, t in pairs]).map(encode, remove_columns=["src", "tgt"])

In [ ]:
from transformers import DataCollatorForSeq2Seq, Trainer, TrainingArguments

args = TrainingArguments(output_dir="lora-out", num_train_epochs=3, per_device_train_batch_size=8, learning_rate=2e-4,
                         warmup_steps=10, logging_steps=10, save_strategy="no", report_to="none", bf16=BF16, fp16=not BF16, seed=0)
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
trainer.train()

In [ ]:
def correct(text):
    messages = [{"role": "user", "content": f"{INSTRUCTION}\n{text}"}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

for s in ["私は図書館を行きました。", "この映画は楽しいでした。", "きれいの花が咲いています。"]:
    print(s, "→", correct(s))

model.save_pretrained("gec-lora")   # 保存されるのは追加した LoRA の重み (数 MB〜数十 MB) だけ

- 学習するパラメータは全体の 1% 未満。元のモデルの重みは固定で、アダプタだけを配布・切り替えできる。
- 読み込み: `PeftModel.from_pretrained(ベースモデル, "gec-lora")`。推論を速くするなら `merge_and_unload()` で重みを統合する。
- 学習率は全体のファインチューニングより高め (1e-4〜3e-4)。`r` は 8〜64 が一般的。
- 指示に従うように調整したあと、応答の好みに合わせてさらに調整する方法は、この章の「選好データで LLM を調整する (DPO)」を参照。
- 4bit 量子化と組み合わせる (QLoRA) と、7B クラスでも T4 で学習できる。`prepare_model_for_kbit_training` を使う。
- ここでは 8 種類の文を 25 回ずつ繰り返しているので、1 エポック目で損失がほぼ 0 になり、学習データを丸暗記する。その結果、「楽しいでした」→「面白かったです」のように、誤りでない語まで学習データの訂正文に寄せて書き換える。実際のデータで学習するときは、検証データの損失を見てエポック数を決める。